# Homebuilder Revenue Forecasting

## Comprehensive Revenue Forecasting for Residential Construction Companies

This notebook demonstrates revenue forecasting for homebuilders like Lennar, DR Horton, Toll Brothers, KB Home, and PulteGroup.

### Key Metrics Covered:
- **Closings**: Home deliveries (revenue recognition)
- **Net Orders**: New contracts signed
- **Backlog**: Homes under contract
- **ASP (Average Selling Price)**: Average home price
- **Absorption Rate**: Sales pace per community
- **Active Communities**: Selling locations
- **Gross Margin**: Profitability
- **Mortgage Rates**: Key external factor
- **Incentives**: Buyer concessions
- **Spec Homes**: Inventory strategy
- **Lot Inventory**: Land pipeline

## 1. Setup and Import

In [ ]:
# Import Revenue Builder
from revenue_builder import RevenueModel
from revenue_builder.business_models import BusinessModelTemplates
from revenue_builder.visualization import Dashboard

import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: f'{x:,.2f}')
plt.style.use('seaborn-v0_8-darkgrid')

print("✓ Libraries imported successfully")

## 2. Generate Sample Homebuilder Data

This simulates 36 months of data including:
- Seasonality (stronger spring/summer)
- Interest rate impacts (2022 rate hikes)
- Backlog dynamics
- Pricing trends
- Margin compression

In [ ]:
# Generate 36 months of sample homebuilder data
sample_data = BusinessModelTemplates.generate_sample_data(
    business_type='homebuilder',
    periods=36,
    start_date='2022-01-01'
)

# Display summary
print(f"Generated {len(sample_data)} months of homebuilder data\n")
print("Data columns:", list(sample_data.columns))
print(f"\nDate range: {sample_data['date'].min()} to {sample_data['date'].max()}")

# Display first few rows
sample_data.head(10)

## 3. Explore Key Homebuilder Metrics

In [ ]:
# Summary statistics for key metrics
key_metrics = [
    'closings', 'net_orders', 'backlog', 'asp', 
    'active_communities', 'absorption_rate', 'gross_margin',
    'mortgage_rate', 'incentive_pct'
]

print("KEY HOMEBUILDER METRICS SUMMARY")
print("=" * 70)
print(sample_data[key_metrics].describe())

## 4. Visualize Historical Trends

In [ ]:
# Create subplots for key metrics
fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('Homebuilder Historical Performance (2022-2024)', fontsize=16, fontweight='bold')

# Closings trend
axes[0, 0].plot(sample_data['date'], sample_data['closings'], 'o-', color='#2E86AB', linewidth=2)
axes[0, 0].set_title('Monthly Closings')
axes[0, 0].set_ylabel('Closings')
axes[0, 0].grid(True, alpha=0.3)

# ASP trend
axes[0, 1].plot(sample_data['date'], sample_data['asp'], 'o-', color='#A23B72', linewidth=2)
axes[0, 1].set_title('Average Selling Price (ASP)')
axes[0, 1].set_ylabel('ASP ($)')
axes[0, 1].grid(True, alpha=0.3)

# Net Orders vs Closings
axes[1, 0].plot(sample_data['date'], sample_data['net_orders'], 'o-', label='Net Orders', linewidth=2)
axes[1, 0].plot(sample_data['date'], sample_data['closings'], 'o-', label='Closings', linewidth=2)
axes[1, 0].set_title('Orders vs Closings')
axes[1, 0].set_ylabel('Units')
axes[1, 0].legend()
axes[1, 0].grid(True, alpha=0.3)

# Backlog
axes[1, 1].plot(sample_data['date'], sample_data['backlog'], 'o-', color='#F18F01', linewidth=2)
axes[1, 1].set_title('Backlog (Units)')
axes[1, 1].set_ylabel('Backlog')
axes[1, 1].grid(True, alpha=0.3)

# Gross Margin
axes[2, 0].plot(sample_data['date'], sample_data['gross_margin'] * 100, 'o-', color='#6A994E', linewidth=2)
axes[2, 0].set_title('Gross Margin %')
axes[2, 0].set_ylabel('Margin (%)')
axes[2, 0].grid(True, alpha=0.3)

# Mortgage Rates Impact
axes[2, 1].plot(sample_data['date'], sample_data['mortgage_rate'], 'o-', color='#E63946', linewidth=2)
axes[2, 1].set_title('30-Year Mortgage Rate')
axes[2, 1].set_ylabel('Rate (%)')
axes[2, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n📊 Historical trends visualized")

## 5. Initialize Homebuilder Revenue Model

In [ ]:
# Initialize homebuilder-specific revenue model
model = RevenueModel(business_type='homebuilder')

print(model.summary())
print("\n✓ Homebuilder model initialized")

## 6. Load and Validate Data

In [ ]:
# Load and validate the homebuilder data
model.load_data(sample_data, validate=True, preprocess=True)

print(f"\n✓ Data loaded successfully")
print(f"Shape: {model.processed_data.shape}")
print(f"Columns: {len(model.processed_data.columns)}")
print(f"\nDate range: {model.processed_data['date'].min()} to {model.processed_data['date'].max()}")

## 7. View Homebuilder Template Details

In [ ]:
# Get homebuilder template
template = BusinessModelTemplates.get_template('homebuilder')

print("HOMEBUILDER BUSINESS MODEL TEMPLATE")
print("=" * 70)
print(f"\nRevenue Model: {template['revenue_model']}")
print(f"Seasonality: {template['seasonality']}")

print("\nRequired Columns:")
for col in template['required_columns']:
    print(f"  • {col}")

print("\nKey Growth Drivers:")
for driver in template['growth_drivers']:
    print(f"  • {driver}")

print("\nExternal Factors:")
for factor in template['external_factors']:
    print(f"  • {factor}")

print("\nRecommended Models:")
for rm in template['recommended_models']:
    print(f"  • {rm}")

## 8. Train Forecasting Models

We'll use multiple approaches:
- **Unit Economics**: Closings × ASP
- **Prophet**: Captures seasonality and trends
- **XGBoost**: Factor-based with mortgage rates, communities, etc.
- **Ensemble**: Combines all models

In [ ]:
# Train multiple models optimized for homebuilders
trained_models = model.train(
    methods=['prophet', 'xgboost'],  # Best for homebuilder forecasting
    target_column='revenue'
)

print(f"\n✓ Trained {len(trained_models)} models")
for model_name in trained_models.keys():
    print(f"  • {model_name}")

## 9. Generate Revenue Forecast

Forecast next 24 months of revenue

In [ ]:
# Generate 24-month forecast
forecast = model.predict(periods=24, confidence_level=0.95)

print("\n✓ Revenue forecast generated")
print(f"\nForecast Summary:")
print(f"Periods: {len(forecast)}")
print(f"Total Forecasted Revenue: ${forecast['forecast'].sum():,.0f}")
print(f"Average Monthly Revenue: ${forecast['forecast'].mean():,.0f}")

# Display first 12 months
print("\nFirst 12 months of forecast:")
forecast.head(12)

## 10. Visualize Revenue Forecast

In [ ]:
# Create dashboard
dashboard = Dashboard()

# Plot revenue forecast
fig = dashboard.plot_revenue_forecast(
    historical=sample_data,
    forecast=forecast,
    date_column='date',
    revenue_column='revenue',
    forecast_column='forecast',
    show_confidence=True,
    title='Homebuilder Revenue Forecast (Historical + 24 Month Projection)'
)

plt.show()

print("\n📈 Revenue forecast visualized")

## 11. Calculate Key Homebuilder Metrics

In [ ]:
# Calculate homebuilder-specific metrics
metrics = model.calculate_metrics()

print("\nKEY HOMEBUILDER METRICS")
print("=" * 70)

# Volume Metrics
print("\nVolume Metrics:")
if 'current_customers' in metrics:
    print(f"  Current Backlog: {metrics.get('current_customers', 0):,.0f} units")
print(f"  Average Monthly Closings: {sample_data['closings'].mean():,.0f}")
print(f"  Average Net Orders: {sample_data['net_orders'].mean():,.0f}")

# Pricing Metrics
print("\nPricing Metrics:")
print(f"  Current ASP: ${sample_data['asp'].iloc[-1]:,.0f}")
print(f"  ASP Growth (YoY): {((sample_data['asp'].iloc[-1] / sample_data['asp'].iloc[0]) - 1) * 100:.1f}%")
print(f"  Current Incentive %: {sample_data['incentive_pct'].iloc[-1] * 100:.1f}%")

# Operational Metrics
print("\nOperational Metrics:")
print(f"  Active Communities: {sample_data['active_communities'].iloc[-1]:,.0f}")
print(f"  Absorption Rate: {sample_data['absorption_rate'].mean():.1f} sales/community/month")
print(f"  Cancellation Rate: {sample_data['cancellation_rate'].mean() * 100:.1f}%")

# Profitability
print("\nProfitability Metrics:")
print(f"  Gross Margin: {sample_data['gross_margin'].mean() * 100:.1f}%")
print(f"  Revenue per Community: ${(sample_data['revenue'] / sample_data['active_communities']).mean():,.0f}")

# Growth Metrics
print("\nGrowth Metrics:")
for key in ['mom_growth_rate', 'yoy_growth_rate', 'cagr']:
    if key in metrics:
        print(f"  {key.replace('_', ' ').title()}: {metrics[key]:.1%}")

## 12. Scenario Analysis: Interest Rate Impact

Model different mortgage rate scenarios and their impact on:
- Order pace
- Cancellations  
- Incentives
- Revenue

In [ ]:
# Run scenario analysis with mortgage rate variations
scenarios = model.scenario_analysis(
    variables={
        # Interest rate scenarios
        # Low rates (5%), base (7%), high rates (8.5%)
    },
    monte_carlo=True,
    n_simulations=5000
)

print(f"\n✓ Generated {len(scenarios)} scenarios")
print("\nScenarios:")
for name in scenarios.keys():
    print(f"  • {name}")

## 13. Compare Scenarios

In [ ]:
# Compare base, optimistic, and pessimistic scenarios
scenario_forecast = {k: v for k, v in scenarios.items() if k != 'monte_carlo'}

if scenario_forecast:
    fig = dashboard.plot_scenario_comparison(
        scenarios=scenario_forecast,
        revenue_column='forecast',
        title='Homebuilder Revenue Scenarios'
    )
    plt.show()
    
    # Summary table
    print("\nScenario Comparison:")
    print("=" * 70)
    for name, df in scenario_forecast.items():
        if 'forecast' in df.columns:
            total_rev = df['forecast'].sum()
            print(f"{name.upper():15s}: ${total_rev:,.0f}")

## 14. Monte Carlo Simulation Results

In [ ]:
# Visualize Monte Carlo distribution
if 'monte_carlo' in scenarios:
    mc_results = scenarios['monte_carlo']
    
    fig = dashboard.plot_monte_carlo_distribution(
        results=mc_results,
        value_column='total_revenue',
        title='Monte Carlo Simulation - 24 Month Revenue Distribution'
    )
    plt.show()
    
    # Print percentiles
    print("\nRevenue Confidence Intervals (24 months):")
    print("=" * 70)
    percentiles = [10, 25, 50, 75, 90]
    for p in percentiles:
        value = mc_results['total_revenue'].quantile(p / 100)
        print(f"P{p:2d}: ${value:,.0f}")

## 15. Backlog-to-Closings Analysis

Understand the conversion from backlog to closings

In [ ]:
# Analyze backlog conversion
sample_data['backlog_months'] = sample_data['backlog'] / sample_data['closings']
sample_data['orders_to_closings_ratio'] = sample_data['net_orders'] / sample_data['closings']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))

# Backlog coverage in months
ax1.plot(sample_data['date'], sample_data['backlog_months'], 'o-', linewidth=2, color='#2E86AB')
ax1.set_title('Backlog Coverage (Months)', fontweight='bold')
ax1.set_ylabel('Months of Backlog')
ax1.axhline(y=5, color='r', linestyle='--', alpha=0.5, label='5 months (typical)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Orders to closings ratio
ax2.plot(sample_data['date'], sample_data['orders_to_closings_ratio'], 'o-', linewidth=2, color='#A23B72')
ax2.set_title('Net Orders to Closings Ratio', fontweight='bold')
ax2.set_ylabel('Ratio')
ax2.axhline(y=1.0, color='r', linestyle='--', alpha=0.5, label='1.0 (breakeven)')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nBacklog Metrics:")
print(f"  Average Backlog Coverage: {sample_data['backlog_months'].mean():.1f} months")
print(f"  Current Backlog Coverage: {sample_data['backlog_months'].iloc[-1]:.1f} months")
print(f"  Orders/Closings Ratio: {sample_data['orders_to_closings_ratio'].mean():.2f}")

## 16. Community Economics Analysis

In [ ]:
# Calculate per-community metrics
sample_data['revenue_per_community'] = sample_data['revenue'] / sample_data['active_communities']

print("COMMUNITY ECONOMICS")
print("=" * 70)
print(f"\nAverage Revenue per Community: ${sample_data['revenue_per_community'].mean():,.0f}/month")
print(f"Average Closings per Community: {sample_data['closings_per_community'].mean():.1f}/month")
print(f"Average Absorption Rate: {sample_data['absorption_rate'].mean():.1f} sales/month")

# Visualize
fig, ax = plt.subplots(figsize=(14, 6))
ax.bar(range(len(sample_data)), sample_data['revenue_per_community'] / 1000000, 
       color='#6A994E', alpha=0.7, edgecolor='black')
ax.set_title('Revenue per Active Community (Monthly)', fontweight='bold', fontsize=14)
ax.set_ylabel('Revenue per Community ($M)')
ax.set_xlabel('Month')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

## 17. Export Comprehensive Report

In [ ]:
# Export comprehensive Excel report
model.export_report(
    path='homebuilder_revenue_forecast_2025.xlsx',
    include_visuals=True,
    include_commentary=True
)

print("\n✓ Comprehensive forecast report exported!")
print("\nReport includes:")
print("  • Historical data")
print("  • 24-month revenue forecast")
print("  • Key homebuilder metrics")
print("  • Scenario analysis")
print("  • Monte Carlo simulation results")
print("  • Executive commentary")

## Summary

This notebook demonstrated:

1. **Homebuilder-Specific Data**: Generated realistic data with seasonality, interest rate impacts, and backlog dynamics
2. **Key Metrics**: Calculated closings, ASP, absorption rates, margins, and more
3. **Forecasting Models**: Trained Prophet and XGBoost models optimized for homebuilders
4. **Scenario Analysis**: Modeled different interest rate and market scenarios
5. **Monte Carlo Simulation**: Generated probabilistic revenue forecasts
6. **Backlog Analysis**: Analyzed conversion from backlog to closings
7. **Community Economics**: Evaluated per-community performance
8. **Comprehensive Reporting**: Exported professional forecast reports

### Key Factors for Homebuilders:
- **Lots**: Land inventory and pipeline (years of supply)
- **Home Prices (ASP)**: Average selling price trends
- **Interest Rates**: Mortgage rate impact on demand
- **Incentives**: Buyer concessions to drive sales
- **Absorption Rate**: Sales pace per community
- **Backlog**: Pipeline of contracted homes
- **Active Communities**: Selling locations
- **Gross Margin**: Profitability trends
- **Seasonality**: Spring/summer stronger than fall/winter
- **Cycle Time**: Months from start to close

Revenue Builder provides a complete solution for homebuilder revenue forecasting!